# Lab 2 — Structured output and a real eval

**~50 minutes.** The step that separates a demo from a system. Ed's courses lean on
structured output throughout; this lab pairs it with the thing most tutorials skip — a
measurement harness, so "I improved the prompt" becomes a number instead of a feeling.

You will:

1. define a schema and force the model into it,
2. handle the failures (there will be failures),
3. score 12 labelled support emails and get a baseline,
4. change one thing, re-score, and see whether you actually helped.

In [ ]:
import json, time
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

from shared import ask, extract_json

emails = json.load(open("../data/support_emails.json"))
print(len(emails), "labelled emails")
print(json.dumps(emails[3], indent=2)[:400])

## 1. The schema is the contract

Free text is unparseable in production. Define the shape first — in the same schema library
you would use for any untrusted input — then make the model fill it in.

Note `needs_human` and `confidence`: designing the schema so *partial success is
expressible* is worth more than any prompt trick. A system that can say "I am not sure,
route this to a person" is one you can actually deploy.

In [ ]:
class Triage(BaseModel):
    # TODO: fields — category (one of: access, billing, how-to, outage, churn,
    # feature-request, compliance, bug, praise, contract), urgency
    # (low|normal|high|critical), needs_human (bool), summary (<= 20 words),
    # confidence (0-1 float).
    pass


print(json.dumps(Triage.model_json_schema(), indent=2))

## 2. Prompt to the schema, validate on receipt, retry once

The pattern, in order:

1. ask, showing the schema and one worked example,
2. parse — the model will sometimes wrap JSON in prose,
3. **validate** against the schema,
4. on failure, retry once with the validation error included; then give up gracefully.

Step 4 is the part people skip. A retry that shows the model its own error message fixes a
surprising share of failures.

In [ ]:
SYSTEM_V1 = """TODO: v1 system prompt — deliberately basic.
Classify a customer support email and reply with JSON matching the schema."""


def triage(email: dict, system: str, temperature: float = 0.0) -> tuple[Triage | None, str]:
    """Return (parsed_or_None, note). `note` records what went wrong, for the eval table."""
    user = f"Subject: {email['subject']}\n\n{email['body']}"
    # TODO: call the model, extract_json, validate with Triage(**data).
    # On ValidationError or ValueError: retry ONCE, appending the error text to the user
    # message, then return (None, "invalid") if it still fails.
    raise NotImplementedError


print(triage(emails[0], SYSTEM_V1))

## 3. The harness

Three numbers, deliberately boring: does it parse, does it agree with the label, and how
often does it need the retry. Everything else in evaluation is a refinement of this.

In [ ]:
def evaluate(system: str, temperature: float = 0.0, label: str = "run") -> dict:
    rows, t0 = [], time.time()
    for email in emails:
        parsed, note = triage(email, system, temperature)
        rows.append({
            "id": email["id"],
            "subject": email["subject"][:38],
            "note": note,
            "category": parsed.category if parsed else "-",
            "cat_ok": bool(parsed and parsed.category == email["label_category"]),
            "urg_ok": bool(parsed and parsed.urgency == email["label_urgency"]),
            "human_ok": bool(parsed and parsed.needs_human == email["label_needs_human"]),
        })
    n = len(rows)
    result = {
        "label": label,
        "schema_valid": sum(r["note"].startswith("ok") for r in rows) / n,
        "category_acc": sum(r["cat_ok"] for r in rows) / n,
        "urgency_acc": sum(r["urg_ok"] for r in rows) / n,
        "needs_human_acc": sum(r["human_ok"] for r in rows) / n,
        "retries": sum(r["note"] == "ok-after-retry" for r in rows),
        "seconds": round(time.time() - t0, 1),
    }
    print(f"{label:12} valid {result['schema_valid']:.0%}  cat {result['category_acc']:.0%}  "
          f"urg {result['urgency_acc']:.0%}  human {result['needs_human_acc']:.0%}  "
          f"retries {result['retries']}  {result['seconds']}s")
    for r in rows:
        flag = " " if r["cat_ok"] else "x"
        print(f"  {flag} {r['id']:2d} {r['subject']:40} -> {r['category']:16} {r['note']}")
    return result


baseline = evaluate(SYSTEM_V1, label="v1 baseline")

## 4. Now improve it — and prove it

Write a v2 system prompt. Things that reliably help, in rough order of payoff:

- **worked examples** (two or three, covering the confusable cases),
- **decision rules for the boundaries** — when is a billing question also compliance? when
  is "urgent" actually critical?,
- **an explicit escape hatch**: what to do when the email fits nothing,
- a reminder to emit JSON only.

Then run the harness again. Keep both numbers. If v2 is worse, that is a result too — it is
the number that stops you shipping a "obvious improvement" that was not one.

In [ ]:
SYSTEM_V2 = """TODO: your improved prompt."""

improved = evaluate(SYSTEM_V2, label="v2 improved")

In [ ]:
for r in (baseline, improved):
    print(f"{r['label']:12} valid {r['schema_valid']:.0%}  cat {r['category_acc']:.0%}  "
          f"urg {r['urgency_acc']:.0%}  human {r['needs_human_acc']:.0%}")

## Stretch goals

1. **Temperature sweep.** Run v2 at 0.0, 0.3 and 0.8. Plot accuracy. Most extraction tasks
   are flat-to-worse above 0.3 — see it for yourself rather than taking the advice.
2. **LLM-as-judge.** The summary field has no ground truth. Write a judge prompt that scores
   each summary 1-3 against a two-line rubric, run it, then read ten summaries yourself and
   check whether you agree with the judge. Calibration before trust.
3. **The confidence field earns its keep.** Compute accuracy on the subset where
   `confidence >= 0.8` versus below. If the model's confidence carries no signal, say so —
   that is a finding, and it decides whether you can route on it.
4. **Add your own cases.** Take five real (redacted) emails from your team, label them, and
   add them to the JSON file. Your eval set is now worth more than this whole notebook.